In [ ]:
"""
  Command for installing the lazyslide package.
"""
!pip install lazyslide

In [ ]:
"""
   Installing openlside package for reading WSI file
"""
!pip install openslide-python==1.4.1

In [ ]:
"""
  Installing other dependencies for openslide
"""
!apt-get update -qq
!apt-get install -y openslide-tools

In [ ]:
"""
  Command for mounting the drive.
"""
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
   Changing the directory to the required working directory. Please change the
   path accordingly where this source file save on the Google drive.
"""
%cd '/content/drive/My Drive/2_PHDWork/1_Objective-I/4_Enlargement_Strategy/1_ComparisonStudy/'

In [ ]:
import os
import csv
from wsidata import open_wsi
from lazyslide.seg import tissue
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from rasterio import features
import time
import cv2 as cv
from scipy.spatial.distance import directed_hausdorff
from PIL import Image
from openslide import OpenSlide

In [ ]:
"""
Writes the comparison data to a csv file.

Parameters:
     log_data: the comparison data to be written.

Returns:
     None
"""
def log_comparison_data(log_data):
  # The name of the log file data
  log_file_name = './log_files/lazyslide_method_results.csv'
  fields = ['dataset','f_name','i_row','i_col','msk_row','msk_col',
            'org_msk_gen_time','m_mem_req','m_accuracy','m_dice','m_iou',
            'm_hdit','m_tpr','m_fpr']

  # Creating a log file if it does not exist.
  if(not os.path.isfile(log_file_name)):
    with open(log_file_name, 'w') as csvfile:
      # creating a csv writer object
      csvwriter = csv.writer(csvfile)
      # writing the fields
      csvwriter.writerow(fields)

  # Writing comparison data to the file.
  try:
    with open(log_file_name,"a") as csvFile:
      writer = csv.DictWriter(csvFile, fieldnames=fields)
      writer.writerow(log_data)
  except IOError:
    print("Writing of Training data failed")

  return

"""
  Method for saving the predicted mask.

  Parameters:
      pred_mask: the predicted mask.
      output_name: the name of the output file.

  Returns:
      None
"""
def save_mask(pred_mask,output_name):
  mask_img = Image.fromarray(pred_mask.astype("uint8") * 255)

  # Save to file
  mask_img.save(output_name)

  return

In [ ]:
"""
  Method for applying the lazyslide tissue segmentation on the given WSI.

  Parameters:
       wsi_path: the location of the WSI file to processed.
       log_data: the log data that needs to be logged.

  Returns:
      mask: the generated mask.
      log_data: the log data that needs to be logged.
"""
def apply_lazyslide_segmentation(wsi_path,log_data):
  wsi = open_wsi(wsi_path)

  start_time = time.process_time()

  # Run tissue segmentation
  tissue(wsi, model="pathprofiler")   # Just updates wsi

  # Get tissue polygons
  tissues = wsi["tissues"]   # GeoDataFrame

  # Slide dimensions
  height, width = wsi._attrs["slide_properties"]["shape"]

  mask = construct_mask(tissues,height, width)

  end_time = time.process_time()
  msk_gen_time = (end_time - start_time)

  log_data['org_msk_gen_time'] = msk_gen_time
  log_data['m_mem_req'] = height * width * 3
  log_data['msk_row'] = height
  log_data['msk_col'] = width

  return mask, log_data

"""
  Method for constructing the binary mask from the lazyslide package utility.

  Parameters:
      tissues: the object returned from the lazyslide package API
      height: the height of the WSI
      width: the width of the WSI

  Returns:
      mask: the binary mask
"""
def construct_mask(tissues,height, width):
  # Rasterize polygons into a binary mask
  mask = features.rasterize(
      [(geom, 1) for geom in tissues.geometry],
      out_shape=(height, width),
      fill=0,
      dtype=np.uint8
    )

  # print(mask.shape)   # (H, W)

  return mask

In [ ]:
"""
  Method for comparing the predicted mask with the ground truth mask.

  Parameters:
      pred_mask       : the predicted mask
      grdtru_msk_file : the ground truth mask file (Expected is png format)
      log_data        : the log data that needs to be logged.
      datst           : the name of the dataset.

  Returns:
      log_data: the log data that needs to be logged.
"""
def perform_ground_truth_analysis(pred_mask,grdtru_msk_file,log_data,datst):
  # The WSI level to be processed
  level = 0
  # Load the ground truth mask
  if(datst == "PANDA"):
    mask_wsi  = OpenSlide(grdtru_msk_file)
    full_mask = np.array(mask_wsi.read_region((0,0),level,
                                              mask_wsi.level_dimensions[level]))
    full_mask = (cv.cvtColor(full_mask,cv.COLOR_RGBA2RGB))[:,:,0]
    grd_mask = np.zeros((full_mask.shape[0],full_mask.shape[1]),dtype=np.uint8)
    grd_mask[full_mask > 0] = 1
  else:
    grd_mask = cv.imread(grdtru_msk_file, cv.IMREAD_GRAYSCALE)
    grd_mask = (grd_mask > 0).astype(np.uint8)

  log_data['i_row'] = grd_mask.shape[0]
  log_data['i_col'] = grd_mask.shape[1]

  accuracy,dice,iou,hausdorff_distance,TPR,FPR = measure_performance_metrics(grd_mask,pred_mask)

  log_data['m_accuracy'] = accuracy
  log_data['m_dice'] = dice
  log_data['m_iou'] = iou
  log_data['m_hdit'] = hausdorff_distance
  log_data['m_tpr'] = TPR
  log_data['m_fpr'] = FPR

  return log_data

"""
Performs a pixel to pixel matching between the given masks and returns the
accuracy, Dice's and Jaccard's coefficient.

Parameters:
     org:     the original labels.
     pred:    the predicted labels.

Returns:
     accuracy:    the percentage of pixels that matches.
     dice:        the Dice's coefficients for tissue pixels.
     iou:         the Intersection over Union for tissue pixels.
     hausdorff_distance: the housdroff distance for tissue pixels.
     TPR:         the true positive rate.
     FPR:         the false positive rate
"""
def measure_performance_metrics(org,pred):
  # Converting the values into 0 and 1.
  i1 = org // 1
  i2 = pred // 1

  # Calculating the TP, TN, FP and FN
  TP = np.sum((i1 * i2))
  TN = np.sum((1 - i1) * (1 - i2))
  FP = np.sum((((i2 - i1) + 1)//2))
  FN = np.sum((((i1 - i2) + 1)//2))

  accuracy = round(((TP + TN)/(TP + TN + FP + FN)),6) * 100

  area_intersection = TP
  area_union = TP + FN + FP

  if(area_union != 0):
    dice = 2 * area_intersection / (np.sum(i1) + np.sum(i2))
    iou = area_intersection / area_union
    # jaccard = area_intersection / area_union
  else:
    dice = 0
    iou = 0

  # Find contours (boundary points) of the masks
  contours1, _ = cv.findContours(org, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
  contours2, _ = cv.findContours(pred, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)

  # Extract boundary points
  points1 = np.vstack(contours1).squeeze()
  points2 = np.vstack(contours2).squeeze()

  u = directed_hausdorff(points1, points2)[0]
  v = directed_hausdorff(points2, points1)[0]
  hausdorff_distance = max(u, v)
  # hausdorff_distance = 0

  TPR = round((TP / (TP + FN)),8)
  FPR = round((FP / (FP + TN)),8)

  return (accuracy,dice,iou,hausdorff_distance,TPR,FPR)

In [ ]:
"""
  The main method where the execution starts.

  Parameters:
       logdata: if true log file is created with all the logs
       savemask: if true the predicted mask is saved.
  Returns: None
"""
def main(logdata=False,savemask=False):
  # List of dataset from which WSIs are considered.
  dataset = ["CPTAC-CM","CPTAC-UCEC","PANDA"]

  # The path location for each dataset.
  dataset_details = {
      "CPTAC-CM" : {"wsi_location"  : "../../0_Dataset/1_Dataset/0_Pure/1_CPTAC-CM/",       # Location where the WSIs are saved.
                    "grdtru_msk_loc": "../../0_Dataset/1_Dataset/2_Masks/1_CPTAC-CM/PNG/",  # Location where the ground truth masks are saved.
                    "output_loc"    : "./0_Generated_Mask/lazyslide_method/1_CPTAC-CM/"     # Location where the predicted masks are to be saved.
                   },
      "CPTAC-UCEC" : {"wsi_location": "../../0_Dataset/1_Dataset/0_Pure/4_CPTAC-UCEC/",
                      "grdtru_msk_loc": "../../0_Dataset/1_Dataset/2_Masks/4_CPTAC-UCEC/PNG/",
                      "output_loc"    : "./0_Generated_Mask/lazyslide_method/4_CPTAC-UCEC/"
                     },
      "PANDA" : {"wsi_location"       : "../../../2_Objective-II/0_Dataset/1_Panda/train_images/radboud/",
                      "grdtru_msk_loc": "../../../2_Objective-II/0_Dataset/1_Panda/train_label_masks/radboud/",
                      "output_loc"    : "./0_Generated_Mask/lazyslide_method/5_PANDA/"
                     }


      }
  ind = 0
  for datst in dataset:
    ind += 1
    """
      Code for avoiding the execution of selected dataset.
    """
    # if(ind != 3):
    #   continue
    print(f'Processing dataset: {datst}')
    wsi_location = dataset_details[datst]["wsi_location"]
    grdtru_msk_loc = dataset_details[datst]["grdtru_msk_loc"]
    output_loc = dataset_details[datst]["output_loc"]

    if(os.path.isdir(wsi_location)):
      files_names = os.listdir(wsi_location)
      print(f'\tProcessing a total of {len(files_names)} WSI files.')

      index = 0
      limit = len(files_names)
      while(index < limit):
        files = files_names[index]
        m_name, ext = os.path.splitext(files)
        wsiname = wsi_location + files
        if(datst == "PANDA"):
          maskfile = grdtru_msk_loc + m_name + "_mask" + ext
        else:
          maskfile = grdtru_msk_loc + m_name + "_mask" + ".png"

        output_name = output_loc + m_name + "_predicted_mask" + ".png"
        log_data = {}
        log_data['dataset'] = datst
        log_data['f_name'] = files
        print(f'\t\t{index + 1} Processing WSI file: {files}')

        pred_mask, log_data = apply_lazyslide_segmentation(wsiname,log_data)
        log_data = perform_ground_truth_analysis(pred_mask,maskfile,log_data,datst)

        if(savemask):
          save_mask(pred_mask,output_name)
        if(logdata):
          log_comparison_data(log_data)
        # break
        index += 1

        del pred_mask, log_data

  return

In [ ]:
main() #Execution starts here